# 04 — Testes do Agente (≥ 5 Cenários)  
**OscaBet Agent** · Validação end-to-end do `predictor.py`

Carrega o modelo treinado e simula o `run_prediction_engine()`  
para os 5 cenários exigidos na entrega.


In [1]:
# IMPORTANTE (Windows): torch ANTES de pandas/numpy (evita WinError 127 / shm.dll).
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import json
from pathlib import Path

# ──────────────────────────────────────────────────────────────────────────────
BASE_DIR   = Path("../..").resolve()
PROC_DIR   = BASE_DIR / "data" / "processed"
MODELS_DIR = BASE_DIR / "agent" / "models"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── Carregar metadados e features ─────────────────────────────────────────────
with open(MODELS_DIR / "oscabet_nn_v1_meta.json") as f:
    meta = json.load(f)

features_df = pd.read_csv(PROC_DIR / "features.csv", parse_dates=["date"])
targets_df  = pd.read_csv(PROC_DIR / "targets.csv",  parse_dates=["match_date"])

FEAT_COLS = meta["feat_cols"]
WINDOW    = meta.get("hyperparams", {}).get("window", 10)

print(f"Features esperadas pelo modelo: {len(FEAT_COLS)}")
print(f"Meta do treinamento: cutoff={meta['train_cutoff']}, n_train={meta['n_train']}")


Features esperadas pelo modelo: 103
Meta do treinamento: cutoff=2024-03-01, n_train=9319


In [2]:
# ── Recarregar arquitetura (IDÊNTICA ao notebook 03 / predictor.py) ────────────
# Heads: resultado=3 classes (H/D/A), cartões=2 (Under/Over), escanteios=2.
class OscaBetNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.backbone = nn.Sequential(
            nn.Linear(input_dim, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.50),
            nn.Linear(256, 128),       nn.BatchNorm1d(128), nn.ReLU(), nn.Dropout(0.40),
            nn.Linear(128, 64),                             nn.ReLU(),
        )
        self.head_result  = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 3))
        self.head_yellow  = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 2))
        self.head_corners = nn.Sequential(nn.Linear(64, 32), nn.ReLU(), nn.Linear(32, 2))

    def forward(self, x):
        h = self.backbone(x)
        return self.head_result(h), self.head_yellow(h), self.head_corners(h)

checkpoint = torch.load(MODELS_DIR / "oscabet_nn_v1.pt", map_location=DEVICE, weights_only=False)
# ENSEMBLE: carrega TODOS os modelos e faz a média das probabilidades (igual ao predictor.py).
_states = checkpoint.get("ensemble", [checkpoint.get("model_state_dict")])
models = []
for _st in _states:
    _m = OscaBetNN(input_dim=meta["input_dim"]).to(DEVICE)
    _m.load_state_dict(_st)
    _m.eval()
    models.append(_m)
print(f"✅ Ensemble carregado ({len(models)} modelo(s))")

softmax = nn.Softmax(dim=1)

def ensemble_probs(x_tensor):
    """Média das probabilidades softmax sobre todos os modelos do ensemble."""
    pr = py = pc = None
    with torch.no_grad():
        for _m in models:
            lr, ly, lc = _m(x_tensor)
            sr = softmax(lr).cpu().numpy()[0]
            sy = softmax(ly).cpu().numpy()[0]
            sc = softmax(lc).cpu().numpy()[0]
            pr = sr if pr is None else pr + sr
            py = sy if py is None else py + sy
            pc = sc if pc is None else pc + sc
    n = len(models)
    return pr / n, py / n, pc / n

# Linhas de mercado (Over/Under) REAIS dos alvos (notebook 02): 4.5 e 9.5.
YELL_LINE = 4.5
CORN_LINE = 9.5


✅ Ensemble carregado (5 modelo(s))


In [3]:
def predict_match(home_team: str, away_team: str, league: str,
                  features_df: pd.DataFrame) -> dict:
    """
    Simula o run_prediction_engine() do tools.py (usando o ENSEMBLE).
    Busca as features mais recentes dos dois times e retorna as probabilidades.
    """
    def last_feats(team, side, league):
        mask = (features_df["league"] == league) & (features_df[f"{side}_team"] == team)
        sub  = features_df[mask].sort_values("date")
        if len(sub) == 0:
            return None
        return sub.iloc[-1]

    home_row = last_feats(home_team, "home", league)
    away_row = last_feats(away_team, "away", league)

    if home_row is None or away_row is None:
        return {"error": f"Time não encontrado: {home_team if home_row is None else away_team}"}

    # Monta o vetor de features mesclando home e away (mesma lógica do predictor.py)
    x_vals = []
    for col in FEAT_COLS:
        if col.startswith("home_") and col in home_row.index:
            val = home_row[col]
        elif col.startswith("away_") and col in away_row.index:
            val = away_row[col]
        elif col.startswith("h2h_") and col in home_row.index:
            val = home_row[col]
        elif col.startswith("league_") and col in home_row.index:
            val = home_row[col]
        else:
            val = 0.5  # fallback neutro
        x_vals.append(0.5 if (val is None or (isinstance(val, float) and np.isnan(val))) else float(val))

    x_tensor = torch.tensor([x_vals], dtype=torch.float32).to(DEVICE)
    pr, py, pc = ensemble_probs(x_tensor)   # média do ensemble

    result_labels = ["H (Mandante)", "D (Empate)", "A (Visitante)"]
    y_labels = [f"Under {YELL_LINE}", f"Over {YELL_LINE}"]
    c_labels = [f"Under {CORN_LINE}", f"Over {CORN_LINE}"]

    pred = {
        "match":       f"{home_team} vs {away_team} ({league})",
        "resultado": {
            "probs":      {l: round(float(p), 3) for l, p in zip(result_labels, pr)},
            "label":      result_labels[int(pr.argmax())],
            "confidence": round(float(pr.max()), 3),
        },
        "cartoes": {
            "probs":      {l: round(float(p), 3) for l, p in zip(y_labels, py)},
            "label":      y_labels[int(py.argmax())],
            "confidence": round(float(py.max()), 3),
        },
        "escanteios": {
            "probs":      {l: round(float(p), 3) for l, p in zip(c_labels, pc)},
            "label":      c_labels[int(pc.argmax())],
            "confidence": round(float(pc.max()), 3),
        },
    }
    return pred


def print_prediction(pred: dict):
    if "error" in pred:
        print(f"❌ Erro: {pred['error']}")
        return
    print(f"\n{'═'*55}")
    print(f"  🏟️  {pred['match']}")
    print(f"{'─'*55}")
    for alvo in ["resultado","cartoes","escanteios"]:
        p = pred[alvo]
        print(f"  {alvo.upper():12s} → {p['label']:20s} (confiança: {p['confidence']*100:.1f}%)")
        for label, prob in p["probs"].items():
            bar = "█" * int(prob * 25)
            print(f"    {label:18s} {prob*100:5.1f}%  {bar}")
    print(f"{'═'*55}")

print("Funções de predição prontas ✅")


Funções de predição prontas ✅


---
## Cenário 1 — Clássico do Brasileirão: Flamengo × Grêmio

**Perfil:** Flamengo como mandante forte (0.72), Grêmio visitante médio (0.59).  
**Esperado:** Vitória do Flamengo favorita, escanteios médio-alto.


In [4]:
pred1 = predict_match("Flamengo", "Grêmio", "brasileirao_a", features_df)
print_prediction(pred1)

# Contexto histórico simulado
h2h_mask = (
    features_df["home_team"].isin(["Flamengo","Grêmio"]) &
    features_df["away_team"].isin(["Flamengo","Grêmio"]) &
    (features_df["league"] == "brasileirao_a")
)
h2h = features_df[h2h_mask].sort_values("date").tail(10)
print(f"\nÚltimos {len(h2h)} confrontos diretos disponíveis no dataset sintético")
print(h2h[["date","home_team","away_team","home_win_rate","away_win_rate"]].tail(5).to_string(index=False))



═══════════════════════════════════════════════════════
  🏟️  Flamengo vs Grêmio (brasileirao_a)
───────────────────────────────────────────────────────
  RESULTADO    → H (Mandante)         (confiança: 72.0%)
    H (Mandante)        72.0%  ██████████████████
    D (Empate)          17.4%  ████
    A (Visitante)       10.6%  ██
  CARTOES      → Under 4.5            (confiança: 53.6%)
    Under 4.5           53.6%  █████████████
    Over 4.5            46.4%  ███████████
  ESCANTEIOS   → Over 9.5             (confiança: 54.7%)
    Under 9.5           45.3%  ███████████
    Over 9.5            54.7%  █████████████
═══════════════════════════════════════════════════════

Últimos 9 confrontos diretos disponíveis no dataset sintético
      date home_team away_team  home_win_rate  away_win_rate
2024-06-13  Flamengo    Grêmio            0.7            0.6
2024-09-22    Grêmio  Flamengo            0.4            0.4
2025-04-13    Grêmio  Flamengo            0.5            0.6
2025-08-31  Flam

---
## Cenário 2 — Premier League: Manchester City × Leeds (Top vs Bottom)

**Perfil:** Campeão (0.76) contra time rebaixado (0.46).  
**Esperado:** Probabilidade altíssima de vitória City, xG elevado para o mandante.


In [5]:
pred2 = predict_match("Manchester City", "Leeds United", "premier_league", features_df)
print_prediction(pred2)



═══════════════════════════════════════════════════════
  🏟️  Manchester City vs Leeds United (premier_league)
───────────────────────────────────────────────────────
  RESULTADO    → H (Mandante)         (confiança: 71.7%)
    H (Mandante)        71.7%  █████████████████
    D (Empate)          17.5%  ████
    A (Visitante)       10.8%  ██
  CARTOES      → Under 4.5            (confiança: 60.9%)
    Under 4.5           60.9%  ███████████████
    Over 4.5            39.1%  █████████
  ESCANTEIOS   → Over 9.5             (confiança: 61.5%)
    Under 9.5           38.5%  █████████
    Over 9.5            61.5%  ███████████████
═══════════════════════════════════════════════════════


---
## Cenário 3 — La Liga: Real Madrid × Getafe (Rivalidade intensa, muitos cartões)

**Perfil:** Real Madrid favoritíssimo, Getafe conhecido por estilo físico.  
**Esperado:** Vitória do Real, cartões amarelos alto ou médio.


In [6]:
pred3 = predict_match("Real Madrid", "Getafe", "la_liga", features_df)
print_prediction(pred3)



═══════════════════════════════════════════════════════
  🏟️  Real Madrid vs Getafe (la_liga)
───────────────────────────────────────────────────────
  RESULTADO    → H (Mandante)         (confiança: 72.5%)
    H (Mandante)        72.5%  ██████████████████
    D (Empate)          17.6%  ████
    A (Visitante)        9.8%  ██
  CARTOES      → Under 4.5            (confiança: 57.5%)
    Under 4.5           57.5%  ██████████████
    Over 4.5            42.5%  ██████████
  ESCANTEIOS   → Over 9.5             (confiança: 56.9%)
    Under 9.5           43.1%  ██████████
    Over 9.5            56.9%  ██████████████
═══════════════════════════════════════════════════════


---
## Cenário 4 — Equilíbrio: Grêmio × Internacional (Gre-Nal)

**Perfil:** Dois times equilibrados, histórico intenso.  
**Esperado:** Resultado mais incerto, empate com probabilidade relevante.


In [7]:
pred4 = predict_match("Grêmio", "Internacional", "brasileirao_a", features_df)
print_prediction(pred4)



═══════════════════════════════════════════════════════
  🏟️  Grêmio vs Internacional (brasileirao_a)
───────────────────────────────────────────────────────
  RESULTADO    → H (Mandante)         (confiança: 38.6%)
    H (Mandante)        38.6%  █████████
    D (Empate)          32.4%  ████████
    A (Visitante)       29.1%  ███████
  CARTOES      → Over 4.5             (confiança: 59.5%)
    Under 4.5           40.5%  ██████████
    Over 4.5            59.5%  ██████████████
  ESCANTEIOS   → Over 9.5             (confiança: 53.5%)
    Under 9.5           46.5%  ███████████
    Over 9.5            53.5%  █████████████
═══════════════════════════════════════════════════════


---
## Cenário 5 — Liga Ofensiva: Liverpool × Manchester City

**Perfil:** Dois dos maiores xG do campeonato. Ambos com muitos escanteios.  
**Esperado:** Escanteios alto, gols esperados elevados, resultado incerto.


In [8]:
pred5 = predict_match("Liverpool FC", "Manchester City", "premier_league", features_df)
print_prediction(pred5)



═══════════════════════════════════════════════════════
  🏟️  Liverpool FC vs Manchester City (premier_league)
───────────────────────────────────────────────────────
  RESULTADO    → A (Visitante)        (confiança: 44.2%)
    H (Mandante)        29.0%  ███████
    D (Empate)          26.8%  ██████
    A (Visitante)       44.2%  ███████████
  CARTOES      → Under 4.5            (confiança: 52.1%)
    Under 4.5           52.1%  █████████████
    Over 4.5            47.9%  ███████████
  ESCANTEIOS   → Over 9.5             (confiança: 55.1%)
    Under 9.5           44.9%  ███████████
    Over 9.5            55.1%  █████████████
═══════════════════════════════════════════════════════


## Sumário dos 5 Cenários

In [9]:
import pandas as pd

rows = []
for pred in [pred1, pred2, pred3, pred4, pred5]:
    if "error" in pred: continue
    rows.append({
        "Partida":          pred["match"],
        "Resultado":        pred["resultado"]["label"],
        "Conf. Resultado":  f"{pred['resultado']['confidence']*100:.1f}%",
        "Cartões":          pred["cartoes"]["label"],
        "Escanteios":       pred["escanteios"]["label"],
    })

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))


                                         Partida     Resultado Conf. Resultado   Cartões Escanteios
              Flamengo vs Grêmio (brasileirao_a)  H (Mandante)           72.0% Under 4.5   Over 9.5
Manchester City vs Leeds United (premier_league)  H (Mandante)           71.7% Under 4.5   Over 9.5
                 Real Madrid vs Getafe (la_liga)  H (Mandante)           72.5% Under 4.5   Over 9.5
         Grêmio vs Internacional (brasileirao_a)  H (Mandante)           38.6%  Over 4.5   Over 9.5
Liverpool FC vs Manchester City (premier_league) A (Visitante)           44.2% Under 4.5   Over 9.5


---
## Próximos Passos

1. **`tools.py`** — adaptar `run_prediction_engine()` para usar este predictor  
2. **`orchestrator.py`** — agentic loop: LLM decide quais tools chamar  
3. **`llm_client.py`** — wrapper da Claude API  
4. **API Flask** — conectar tudo via `/api/chat`  
5. **Frontend React** — `PredictionCard` renderizado quando NN é acionada
